# Basic Data Transformation

## Definition

**Data transformation** is the process of converting data from one format, structure, or scale into another so that it is suitable for **analysis, visualization, or machine learning**.

In Python, **Pandas** provides efficient tools for creating features, transforming values, reshaping data, and working with categorical variables.

---

## Why It Matters

Real-world data rarely comes in the exact form required for analysis or machine learning.

For example:

* A date may need to be split into **year, month, or day**.
* A metric such as **profit margin** may need to be calculated from existing columns.
* Text categories may need to be converted into a format that algorithms can process.
* Continuous values may need to be divided into meaningful groups.

Data transformation converts existing data into a more **useful and analysis-ready form**.

> **Data cleaning fixes problems in data; data transformation changes data into a more useful form.**

---

## How It Works

Common data transformation tasks include:

* **Feature Engineering**: Creating new columns from existing data using mathematical operations, conditions, or custom functions.
* **Value Mapping**: Replacing or converting values using dictionaries or lookup logic.
* **Discretization (Binning)**: Converting continuous numerical values into discrete categories or ranges.
* **Categorical Encoding**: Converting categorical values into numerical representations for machine learning.
* **Data Scaling**: Transforming numerical features to a common scale so that differences in magnitude do not disproportionately affect certain algorithms.

---

## Common Data Transformation Actions

| Transformation           | Purpose                                           | Example                          |
| ------------------------ | ------------------------------------------------- | -------------------------------- |
| **Feature Engineering**  | Create meaningful features from existing data     | `Profit / Sales` → Profit Margin |
| **Value Mapping**        | Replace or remap existing values                  | `"Y"` → `"Yes"`                  |
| **Discretization**       | Convert continuous values into groups             | Age → Young / Adult / Senior     |
| **Categorical Encoding** | Convert categories into numerical representations | `"Male"` → `0`, `"Female"` → `1` |
| **Data Scaling**         | Bring numerical features to a comparable scale    | `0–100000` → `0–1`               |
| **Date Transformation**  | Extract useful information from dates             | Date → Year / Month / Quarter    |

---

## Common Mistakes

### 1. Scaling Before Splitting the Data

In machine learning, scaling the entire dataset **before** splitting it into training and testing sets can cause **data leakage**.

The scaler learns information such as the mean, standard deviation, minimum, or maximum from the test data.

**Correct approach:**

```text
Split Data
    ↓
Fit scaler on Training Data
    ↓
Transform Training Data
    ↓
Transform Test Data using the same scaler
```

---

### 2. Overwriting Original Features

Replacing the original column directly can make it difficult to validate the transformation or recover the original values.

Prefer creating a new column when the original data may still be useful:

```python
df["Profit Margin"] = df["Profit"] / df["Sales"]
```

---

### 3. Creating Unintended Missing Values

When using a mapping dictionary, values that do not have a corresponding key can become missing (`NaN`).

```python
df["Category"] = df["Category"].map({
    "A": "High",
    "B": "Medium"
})
```

If the original column contains `"C"`, it will become `NaN`.

Always verify that all expected values are handled.

---

### 4. Transforming Data Without a Purpose

Not every possible transformation is useful.

A transformation should have a clear **analytical or business purpose**.

For example:

```python
df["Profit Margin"] = df["Profit"] / df["Sales"]
```

is useful because it helps answer:

> Which products generate high sales but relatively low profit?

---

## Quick Recap

* **Data transformation** converts existing data into a more useful form for analysis or machine learning.
* **Pandas** provides tools for creating features, mapping values, binning data, encoding categories, and manipulating dates.
* Common transformations include **feature engineering, value mapping, discretization, encoding, scaling, and date transformation**.
* Transformations should have a clear **analytical or business purpose**.
* In machine learning, **split the data before fitting transformations that learn from the data** to prevent data leakage.
* Keep original columns when they are useful for **validation, comparison, or reference**.


In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data/retail_store_sales_cleaned.csv", parse_dates=["Transaction Date"])

In [3]:
df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10,185.0,Digital Wallet,Online,2024-04-08
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9,261.0,Digital Wallet,Online,2023-07-23
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2,43.0,Credit Card,Online,2022-10-05
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9,247.5,Credit Card,Online,2022-05-07
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7,87.5,Digital Wallet,Online,2022-10-02


In [4]:
print(f"Rows: {df.shape[0]}\nColumns: {df.shape[1]}")

Rows: 11971
Columns: 10


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11971 entries, 0 to 11970
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    11971 non-null  str           
 1   Customer ID       11971 non-null  str           
 2   Category          11971 non-null  str           
 3   Item              11971 non-null  str           
 4   Price Per Unit    11971 non-null  float64       
 5   Quantity          11971 non-null  int64         
 6   Total Spent       11971 non-null  float64       
 7   Payment Method    11971 non-null  str           
 8   Location          11971 non-null  str           
 9   Transaction Date  11971 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(2), int64(1), str(6)
memory usage: 935.4 KB


### Feature Engineering

In [6]:
df["Customer Total Spending"] = (
    df.groupby("Customer ID")["Total Spent"].transform("sum")
)

df["Avg Order Value"] = (
    df.groupby("Customer ID")["Total Spent"].transform("mean")
)

### Binning

In [7]:
sales_labels = ["Low", "Medium", "High"]
quantity_labels = ["Low", "Standard", "Bulk"]
df["Sales Quartile"] = pd.qcut(
    x=df["Total Spent"],
    q=3,
    labels=sales_labels,
)

df["Quantity Quartile"] = pd.qcut(
    x=df["Quantity"],
    q=3,
    labels=quantity_labels
)

### Date Transformation

In [8]:
df["Transaction Year"] = df["Transaction Date"].dt.year
df["Transaction Month"] = df["Transaction Date"].dt.month
df["Transaction Month Name"] = df["Transaction Date"].dt.month_name()
df["Transaction Quarter"] = df["Transaction Date"].dt.quarter
df["Transaction Weekday"] = df["Transaction Date"].dt.weekday
df["Transaction Weekday Name"] = df["Transaction Date"].dt.day_name()

df["Is Weekend"] = df["Transaction Weekday"].isin([5,6]).map({
    True: "Yes",
    False: "No"
})

In [9]:
df.columns

Index(['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit',
       'Quantity', 'Total Spent', 'Payment Method', 'Location',
       'Transaction Date', 'Customer Total Spending', 'Avg Order Value',
       'Sales Quartile', 'Quantity Quartile', 'Transaction Year',
       'Transaction Month', 'Transaction Month Name', 'Transaction Quarter',
       'Transaction Weekday', 'Transaction Weekday Name', 'Is Weekend'],
      dtype='str')

In [10]:
print(f"Rows: {df.shape[0]}\nColumns: {df.shape[1]}")

Rows: 11971
Columns: 21


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11971 entries, 0 to 11970
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Transaction ID            11971 non-null  str           
 1   Customer ID               11971 non-null  str           
 2   Category                  11971 non-null  str           
 3   Item                      11971 non-null  str           
 4   Price Per Unit            11971 non-null  float64       
 5   Quantity                  11971 non-null  int64         
 6   Total Spent               11971 non-null  float64       
 7   Payment Method            11971 non-null  str           
 8   Location                  11971 non-null  str           
 9   Transaction Date          11971 non-null  datetime64[us]
 10  Customer Total Spending   11971 non-null  float64       
 11  Avg Order Value           11971 non-null  float64       
 12  Sales Quartile            119

In [12]:
# Core Transaction Information
transaction_cols = [
    "Transaction ID",
    "Customer ID",
    "Category",
    "Item",
    "Price Per Unit",
    "Quantity",
    "Total Spent",
    "Payment Method",
    "Location",
    "Transaction Date"
]

# Time-Based Features
time_cols = [
    "Transaction Year",
    "Transaction Month",
    "Transaction Month Name",
    "Transaction Quarter",
    "Transaction Weekday",
    "Transaction Weekday Name",
    "Is Weekend"
]

# Analytical Features
analysis_cols = [
    "Sales Quartile",
    "Quantity Quartile",
    "Customer Total Spending",
    "Avg Order Value"
]

# Reorder Columns
df = df.loc[:, transaction_cols + time_cols + analysis_cols]

In [13]:
df.to_csv("data/retail_store_sales_transformed.csv", index=False)

In [14]:
print("End of Day 4")

End of Day 4
